# Brain Tumor Detection (Colab recreation, non-Kaggle dataset)

Recreation of the local `Brain-tumor-prediction` project using a real, citable academic dataset instead of an unattributed Kaggle repost.

**Dataset**: Multi-Class Brain Tumor MRI Dataset (Glioma, Healthy Brain, Meningioma, Pituitary Macroadenoma) — Mendeley Data
- Citation: Jamil, Nusrat; Khan, Abbas Ali (2025), "Multi-Class Brain Tumor MRI Dataset: Glioma, Healthy Brain, Meningioma, and Pituitary Macroadenoma", Mendeley Data, V2, doi: [10.17632/82mtzd8x72.2](https://data.mendeley.com/datasets/82mtzd8x72/2)
- License: CC BY 4.0
- 1,137 MRI images across 4 classes, split 80/20 stratified into `Training/` (908 images) and `Testing/` (229 images), each with `Glioma/`, `Healthy/`, `Meningioma/`, `Pituitary Macroadenoma/` subfolders.
- Already converted and zipped locally as `train.zip`/`test.zip` — just upload both to Google Drive (see step 1 below).

**Before running**:
1. Upload `train.zip` and `test.zip` (already prepared in this project folder) to `MyDrive/BrainTumorMRI/` in Google Drive.
2. Run the cells below in order with a GPU runtime (Runtime → Change runtime type → T4 GPU).

## Mount Drive and unzip the dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/BrainTumorMRI'

import os
assert os.path.exists(f'{DRIVE_DIR}/train.zip'), f'Upload train.zip to {DRIVE_DIR}/ first'
assert os.path.exists(f'{DRIVE_DIR}/test.zip'), f'Upload test.zip to {DRIVE_DIR}/ first'

In [ ]:
!mkdir -p /content/Training /content/Testing
!unzip -q "$DRIVE_DIR/train.zip" -d /content/Training
!unzip -q "$DRIVE_DIR/test.zip" -d /content/Testing

# If the zip contains a nested top-level folder (e.g. Training/train/Glioma/...),
# adjust train_dir/test_dir below to point at the folder that directly contains
# the four class subfolders.
train_dir = '/content/Training'
test_dir = '/content/Testing'
print(os.listdir(train_dir))
print(os.listdir(test_dir))

## 💌 Read Dataset

In [ ]:
from sklearn.utils import shuffle

train_paths = []
train_labels = []

for label in os.listdir(train_dir):
    label_folder = os.path.join(train_dir, label)
    if os.path.isdir(label_folder):
        for image in os.listdir(label_folder):
            train_paths.append(os.path.join(label_folder, image))
            train_labels.append(label)

train_paths, train_labels = shuffle(train_paths, train_labels)

print(f"Total training images: {len(train_paths)}")
print(f"Example: {train_paths[0]}  -->  Label: {train_labels[0]}")

test_paths = []
test_labels = []

for label in os.listdir(test_dir):
    label_folder = os.path.join(test_dir, label)
    if os.path.isdir(label_folder):
        for image in os.listdir(label_folder):
            test_paths.append(os.path.join(label_folder, image))
            test_labels.append(label)

test_paths, test_labels = shuffle(test_paths, test_labels)

print(f"Total testing images: {len(test_paths)}")
print(f"Example: {test_paths[0]}  -->  Label: {test_labels[0]}")

## 📦 Imports

In [ ]:
import numpy as np
import random
from PIL import Image, ImageEnhance
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Flatten, Dropout
from tensorflow.keras.preprocessing.image import load_img
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications import VGG16

IMAGE_SIZE = 128

## 📊 Data Visualization

In [ ]:
random_indices = random.sample(range(len(train_paths)), 10)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.ravel()

for i, idx in enumerate(random_indices):
    img_path = train_paths[idx]
    img = Image.open(img_path)
    img = img.resize((128, 128))
    axes[i].imshow(img)
    axes[i].axis("off")
    axes[i].set_title(f"Label: {train_labels[idx]}", fontsize=14)
plt.tight_layout()
plt.show()

## 🎥 Image Preprocessing

In [ ]:
def augment_image(image):
    image = Image.fromarray(np.uint8(image))
    image = ImageEnhance.Brightness(image).enhance(random.uniform(0.8, 1.2))
    image = ImageEnhance.Contrast(image).enhance(random.uniform(0.8, 1.2))
    if random.random() < 0.5:
        image = image.transpose(Image.FLIP_LEFT_RIGHT)
    image = image.rotate(random.uniform(-15, 15), fillcolor=(0, 0, 0))
    image = np.array(image) / 255.0
    return image

def open_images(paths, augment=False):
    images = []
    for path in paths:
        image = load_img(path, target_size=(IMAGE_SIZE, IMAGE_SIZE))
        if augment:
            image = augment_image(image)
        else:
            image = np.array(image) / 255.0
        images.append(image)
    return np.array(images)

def encode_label(labels):
    unique_labels = sorted(os.listdir(train_dir))
    encoded = [unique_labels.index(label) for label in labels]
    return np.array(encoded)

def datagen(paths, labels, batch_size=12, epochs=1):
    for _ in range(epochs):
        for i in range(0, len(paths), batch_size):
            batch_paths = paths[i:i + batch_size]
            batch_images = open_images(batch_paths, augment=True)
            batch_labels = labels[i:i + batch_size]
            batch_labels = encode_label(batch_labels)
            yield batch_images, batch_labels

## GPU Check

In [ ]:
import tensorflow as tf

if tf.config.list_physical_devices('GPU'):
    print("✅ GPU is available!")
else:
    print("⚠️  GPU not available — set Runtime > Change runtime type > T4 GPU.")

## Model Architecture

In [ ]:
base_model = VGG16(input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3), include_top=False, weights='imagenet')

for layers in base_model.layers:
    layers.trainable = False

base_model.layers[-2].trainable = True
base_model.layers[-3].trainable = True
base_model.layers[-4].trainable = True

model = Sequential()
model.add(Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3)))
model.add(base_model)
model.add(Flatten())
model.add(Dropout(0.3))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.2))

model.add(Dense(len(os.listdir(train_dir)), activation='softmax'))

model.compile(optimizer=Adam(learning_rate=0.0001), loss='sparse_categorical_crossentropy', metrics=['sparse_categorical_accuracy'])

# Carve a stratified validation split out of the training data (Testing/ stays untouched for final reporting)
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

train_paths_fit, val_paths, train_labels_fit, val_labels = train_test_split(
    train_paths, train_labels, test_size=0.15, stratify=train_labels, random_state=42
)
print(f"Train: {len(train_paths_fit)}  Val: {len(val_paths)}")

val_images = open_images(val_paths, augment=False)
val_labels_encoded = encode_label(val_labels)

batch_size = 20
steps = int(len(train_paths_fit) / batch_size)
epochs = 30

early_stop = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

history = model.fit(
    datagen(train_paths_fit, train_labels_fit, batch_size=batch_size, epochs=epochs),
    validation_data=(val_images, val_labels_encoded),
    epochs=epochs,
    steps_per_epoch=steps,
    callbacks=[early_stop]
)

## Training Curves (train vs validation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['sparse_categorical_accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_sparse_categorical_accuracy'], label='Val Accuracy')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## Model Saving (to Google Drive)

In [ ]:
import pickle

model.save(f'{DRIVE_DIR}/Brain_Tumor_Prediction.h5')

with open(f'{DRIVE_DIR}/training_history.pkl', 'wb') as f:
    pickle.dump(history.history, f)

print(f'Saved model and history to {DRIVE_DIR}')

## Model Classification Report

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import seaborn as sns

test_images = open_images(test_paths)
test_labels_encoded = encode_label(test_labels)

test_predictions = model.predict(test_images)

print("Classification Report:")
print(classification_report(test_labels_encoded, np.argmax(test_predictions, axis=1)))

## Confusion Matrix

In [ ]:
conf_matrix = confusion_matrix(test_labels_encoded, np.argmax(test_predictions, axis=1))
print("Confusion Matrix:")
print(conf_matrix)

plt.figure(figsize=(8, 6))
sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues", xticklabels=sorted(os.listdir(train_dir)), yticklabels=sorted(os.listdir(train_dir)))
plt.title("Confusion Matrix")
plt.xlabel("Predicted Labels")
plt.ylabel("True Labels")
plt.show()

## ROC Curve

_Note: AUC here is one-vs-rest and measures ranking quality across every threshold, while the accuracy above is measured at a single threshold (argmax). A model can have high AUC per class while overall accuracy is lower — that is not a contradiction._

In [ ]:
class_names = sorted(os.listdir(train_dir))
num_classes = len(class_names)
test_labels_bin = label_binarize(test_labels_encoded, classes=np.arange(num_classes))
test_predictions_bin = test_predictions

fpr, tpr, roc_auc = {}, {}, {}
for i in range(num_classes):
    fpr[i], tpr[i], _ = roc_curve(test_labels_bin[:, i], test_predictions_bin[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(10, 8))
for i in range(num_classes):
    plt.plot(fpr[i], tpr[i], label=f'{class_names[i]} (AUC = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title("ROC Curve")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.show()

## MRI Tumor Detection System (for new images)

In [ ]:
from tensorflow.keras.preprocessing.image import load_img, img_to_array

# Sorted so the class->index mapping is stable across sessions/reruns
class_labels = sorted(os.listdir(train_dir))
print(class_labels)

def detect_and_display(img_path, model, image_size=128):
    try:
        img = load_img(img_path, target_size=(image_size, image_size))
        img_array = img_to_array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)

        predictions = model.predict(img_array)
        predicted_class_index = np.argmax(predictions, axis=1)[0]
        confidence_score = np.max(predictions, axis=1)[0]

        predicted_label = class_labels[predicted_class_index]
        result = f"Tumor: {predicted_label}"

        plt.imshow(load_img(img_path))
        plt.axis('off')
        plt.title(f"{result} (Confidence: {confidence_score * 100:.2f}%)")
        plt.show()

    except Exception as e:
        print("Error processing the image:", str(e))

In [ ]:
# Example usage — pick any real file from your unzipped Testing folders
sample_path = test_paths[0]
print(f"True label: {test_labels[0]}")
detect_and_display(sample_path, model)